Visualize images from pre-processing steps: 

In [9]:
import SimpleITK as sitk  
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from pathlib import Path
import ipywidgets as widgets
import pandas as pd
from monai.transforms import RemoveSmallObjects, KeepLargestConnectedComponent
import blosc2
# ============================================================
# Utility Functions
# ============================================================

def load_b2nd(path):
    """Load a .b2nd file as a numpy array using blosc2."""
    schunk = blosc2.open(path, mode="r")
    arr = schunk[:][0]  # Load full array
    return arr

def load_volume(path):
    """Load a 3D volume from NIfTI (.nii.gz), NPZ, NPY, or B2ND files."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    if path.suffix == ".npy":
        return np.load(path)
    elif path.suffix == ".npz":
        data = np.load(path)
        key = list(data.keys())[0]
        
        return data[key]
    elif path.suffix in [".nii", ".gz"]:
        return  nib.load(str(path)).get_fdata()
    elif path.suffix == ".b2nd":
        return load_b2nd(path)
    else:
        raise ValueError(f"Unsupported format: {path.suffix}")

def resample_label_to_image(label_path, image_ref_path):
    """Resample label to match reference image (CT or input image)."""
    label_sitk = sitk.ReadImage(str(label_path))
    ref_sitk = sitk.ReadImage(str(image_ref_path))
    label_resampled = sitk.Resample(
        label_sitk,
        ref_sitk,
        sitk.Transform(),
        sitk.sitkNearestNeighbor,
        0,
        label_sitk.GetPixelID()
    )
    print(f"Resampled label: {label_resampled.GetSize()} to match CT: {ref_sitk.GetSize()}")
    return sitk.GetArrayFromImage(label_resampled).transpose(2, 1, 0)

def window_ct_hu(ct_hu, level=50, width=350):
    lower, upper = level - width / 2.0, level + width / 2.0
    ct_clipped = np.clip(ct_hu, lower, upper)
    return (ct_clipped - lower) / (upper - lower + 1e-6)

def get_slice(volume, axis, idx):
    if axis == "axial":
        return volume[idx, :, :]
    elif axis == "coronal":
        return volume[:, idx, :]
    elif axis == "sagittal":
        return volume[:, :, idx]
    else:
        raise ValueError(f"Invalid axis: {axis}. Must be 'axial', 'coronal', or 'sagittal'.")

# ============================================================
# Overlay Builders
# ============================================================

def build_label_overlay(label_slice):
    unique_labels = np.unique(label_slice)
    if np.array_equal(unique_labels, [0]) or np.array_equal(unique_labels, [0, 1]):
        cmap = ListedColormap([[0,0,0,0], [1,0,0,0.2]])
        return label_slice.astype(np.int32), cmap, (0, 1)
    colors = [[0, 0, 0, 0]]
    rng = np.random.default_rng(42)
    for _ in range(int(unique_labels.max())):
        colors.append([*rng.random(3), 0.2])
    cmap = ListedColormap(colors)
    return label_slice.astype(np.int32), cmap, (0, int(unique_labels.max()))

def build_label_overlay_small_objects_3d(label_volume, min_size=200, connectivity=2):
    delete_small = RemoveSmallObjects(min_size=min_size, connectivity=connectivity)
    cleaned_label = delete_small(label_volume)
    small_objects = (label_volume > 0) & (cleaned_label == 0)
    large_objects = (cleaned_label > 0)
    overlay = np.zeros_like(label_volume, dtype=np.int32)
    overlay[large_objects] = 1
    overlay[small_objects] = 2
    cmap = ListedColormap([[0,0,0,0],[0,1,0,0.25],[1,0,0,0.25]])
    return overlay, cmap, (0, 2)

def build_label_overlay_largest_3d(label_volume):
    largest_component = KeepLargestConnectedComponent(connectivity=2)(label_volume)
    overlay = np.zeros_like(label_volume, dtype=np.int32)
    overlay[label_volume > 0] = 1
    overlay[largest_component > 0] = 2
    cmap = ListedColormap([[0,0,0,0],[1,0,0,0.25],[0,1,0,0.25]])
    return overlay, cmap, (0, 2)

# ============================================================
# Visualization
# ============================================================

def visualize_case(ct_path, label_path, uid, target,
                   axis="axial", mode="labels",
                   window_level=50, window_width=350,
                   save_dir=None, save_all_slices=False,
                   resample_labels=False):
    """
    Visualize one case with selectable mode:
      - 'labels', 'large_component', 'small_objects'
    Optionally resample labels to match CT grid.
    """
    ct = load_volume(ct_path)
    if resample_labels:
        label = resample_label_to_image(label_path, ct_path)
    else:
        label = load_volume(label_path)
    print(ct_path.suffix)
    if not ct_path.suffix == ".b2nd":
        ct = np.transpose(ct, (2, 1, 0)) 
        label = np.transpose(label, (2, 1, 0)) 
        
    print(f"Loaded UID {uid}: CT shape {ct.shape}, Label shape {label.shape}")
    assert ct.shape == label.shape, f"Shape mismatch for {uid}"
    
    if axis == "axial":
        reduce_axes = (1, 2)  
    elif axis == "coronal":
        reduce_axes = (0, 2)  
    elif axis == "sagittal":
        reduce_axes = (0, 1)  

    annotated_slices= np.where(np.any(label > 0, axis=reduce_axes))[0]
    
    if len(annotated_slices) > 0:
        print(f"Annotations at slices: {annotated_slices}")
    else:
        print("No annotations found.")

    if mode == "large_component":
        overlay_3d, cmap, vminmax = build_label_overlay_largest_3d(label)
    elif mode == "small_objects":
        overlay_3d, cmap, vminmax = build_label_overlay_small_objects_3d(label)
    else:
        overlay_3d, cmap, vminmax = label, *build_label_overlay(label[:, :, 0])[1:]

    axis_to_dim = {"axial": 0, "coronal": 1, "sagittal": 2}
    n_slices = ct.shape[axis_to_dim[axis]]

    out_uid_dir = None
    if save_all_slices and save_dir:
        out_uid_dir = Path(save_dir) / str(uid)
        out_uid_dir.mkdir(parents=True, exist_ok=True)

    def plot_slice(idx):
        ct_slice = get_slice(ct, axis, idx)
        overlay_slice = get_slice(overlay_3d, axis, idx)
        ct_img = window_ct_hu(ct_slice, window_level, window_width)

        plt.figure(figsize=(6,6))
        plt.imshow(ct_img, cmap="gray", origin="lower")
        plt.imshow(overlay_slice, cmap=cmap, origin="lower",
                   vmin=vminmax[0], vmax=vminmax[1])
        plt.axis("off")
        plt.title(f"UID {uid} | {target} | {axis.capitalize()} slice {idx}/{n_slices}")
        plt.tight_layout()

        if out_uid_dir:
            plt.savefig(out_uid_dir / f"{axis}_slice{idx:03d}.png", bbox_inches='tight')
            plt.close()
        else:
            plt.show()

    slice_slider = widgets.IntSlider(
        value=n_slices//2, min=0, max=n_slices-1, step=1,
        description=f'UID {uid}', continuous_update=False
    )
    widgets.interact(plot_slice, idx=slice_slider)

    if save_all_slices and out_uid_dir:
        print(f"Saving all slices for UID {uid} → {out_uid_dir}")
        for i in range(n_slices):
            plot_slice(i)

# ============================
# Main Visualization Function
# ============================


def visualize_dataset(uids, img_root, label_root,resample_labels=False,mode="label", axis="axial", save_dir=None, save_all_slices=False):
    """Visualize multiple UIDs."""
    img_root, label_root = Path(img_root), Path(label_root)
    df = pd.read_csv(f"/data/colon_cancer/Classifier/ColonCancer/splits.csv")
    for uid in uids:
        # try naming patterns
        possible_img_names = [
            f"{uid}.nii.gz",f"{uid:03d}.nii.gz", f"{uid}_0000.nii.gz", f"{uid:03d}_0000.nii.gz",
            f"colon_{uid:03d}.nii.gz", f"{uid}.npy", f"{uid}.npz",f"{uid:03d}.npz",  f"{uid:03d}.b2nd",
        ]
        possible_label_names = [
            f"{uid}.nii.gz",f"{uid:03d}.nii.gz", f"{uid}_0000.nii.gz", f"{uid:03d}_0000.nii.gz",
            f"{uid:03d}_seg.nii.gz",f"{uid}_seg.nii.gz", f"{uid}.npy", f"{uid}.npz",f"{uid:03d}_seg.npz",  f"{uid:03d}_seg.b2nd", 
        ]
        
        for n in possible_img_names:
            print(f"     {img_root / n} -> {'✅ exists' if (img_root / n).exists() else '❌ missing'}")
        for n in possible_label_names:
            print(f"     {label_root / n} -> {'✅ exists' if (label_root / n).exists() else '❌ missing'}")
            
        img_path = next((img_root / n for n in possible_img_names if (img_root / n).exists()), None)
        label_path = next((label_root / n for n in possible_label_names if (label_root / n).exists()), None)
       
        if img_path is None or label_path is None:
            print(f"Skipping UID {uid}: missing image or label")
            continue

        row = df[df["UID"] == uid]
        target = row["target"].values[0]
        if target == 0:
            target = "div"
        else:
            target = "cc"

        visualize_case(img_path, label_path, uid,mode=mode, axis=axis, target=target,
                       save_dir=save_dir, save_all_slices=save_all_slices,resample_labels=resample_labels)


# ============================
# Example usage
# ============================

if __name__ == "__main__":
    #108, 565, 752, 803, 370
    """
    #finetuned model output
    uids = [16  ]
    img_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/"
    label_root = "/data/colon_cancer/nnUNet_results/Dataset101_CC/nnUNetTrainer__nnUNetResEncUNetLPlans__3d_fullres/fold_0/validation"
    save_dir = "/data/benchaaben/classifier/ct_overlays/finetuned"

    visualize_dataset(uids, img_root, label_root,
                    axis="axial", save_dir=save_dir, mode ="label", save_all_slices=False)
    
    uids = [277 ]
    #resampled refined labels
    img_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/"
    label_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/labels/final"
    save_dir = "/data/benchaaben/classifier/ct_overlays/pretrained"
    visualize_dataset(uids, img_root, label_root,
                      axis="axial", save_dir=save_dir,mode ="label",resample_labels=True, save_all_slices=False)

    #output of pretrained segmentation model
    img_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/"
    label_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/labels/original"
    save_dir = "/data/benchaaben/classifier/ct_overlays/pretrained"
    visualize_dataset(uids, img_root, label_root,mode ="label",
                      axis="axial", save_dir=save_dir, save_all_slices=False)
    
    #original GT
    img_root = "/data/colon_cancer/nnUNet_raw/Dataset100_CC/imagesTr"
    label_root = "/data/colon_cancer/nnUNet_raw/Dataset100_CC/labelsTr"
    save_dir = "/data/benchaaben/classifier/ct_overlays/original"
    visualize_dataset(uids, img_root, label_root,mode ="label",
                      axis="axial", save_dir=save_dir, save_all_slices=False)
    """
    """
    uids = [277 ]
    ##input to segmentator
    img_root = f"/data/colon_cancer/Classifier/ColonCancer/nnUNetPlans_3d_fullres"
    label_root = f"/data/colon_cancer/Classifier/ColonCancer/nnUNetPlans_3d_fullres"
    visualize_dataset(uids, img_root, label_root,mode ="label",
                      axis="axial", save_all_slices=False)
    
    ##input to classifier 
    uids = [277]
    img_root = f"/data/colon_cancer/CC_Detection/pp_data/Dataset100_CC/rescaledTr"
    label_root = f"/data/colon_cancer/CC_Detection/pp_data/Dataset100_CC/resampledTr/labels_resampled"
    visualize_dataset(uids, img_root, label_root,mode ="label",
                      axis="axial", save_all_slices=False)
    """
    uids = [2 ]
    img_root = "/data/colon_cancer/nnUNet_raw/Dataset102_CC/imagesTr"
    label_root = "/data/colon_cancer/nnUNet_raw/Dataset102_CC/labelsTr"
    save_dir = "/data/benchaaben/classifier/ct_overlays/finetuned"
    visualize_dataset(uids, img_root, label_root,
                    axis="axial", save_dir=save_dir, mode ="label", save_all_slices=False)
    
    img_root = "/data/colon_cancer/nnUNet_preprocessed/Dataset102_CC/nnUNetPlans_3d_fullres"
    label_root = "/data/colon_cancer/nnUNet_preprocessed/Dataset102_CC/nnUNetPlans_3d_fullres"
    save_dir = "/data/benchaaben/classifier/ct_overlays/pretrained"
    visualize_dataset(uids, img_root, label_root,mode ="label",
                      axis="axial", save_dir=save_dir, resample_labels=False, save_all_slices=False)
    
    """
    uids = [380 ]
    img_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/"
    label_root = "/data/colon_cancer/labelsTr_finetuned_all_folds"
    save_dir = "/data/benchaaben/classifier/ct_overlays/finetuned"
    visualize_dataset(uids, img_root, label_root,
                    axis="axial", save_dir=save_dir, mode ="label", save_all_slices=False)
    
    img_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/"
    label_root = "/data/colon_cancer/labelsTr_finetuned_finetuned_012"
    save_dir = "/data/benchaaben/classifier/ct_overlays/finetuned"
    visualize_dataset(uids, img_root, label_root,
                    axis="axial", save_dir=save_dir, mode ="label", save_all_slices=False)
    
  
    
    img_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/"
    label_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/labels/original"
    save_dir = "/data/benchaaben/classifier/ct_overlays/pretrained"
    visualize_dataset(uids, img_root, label_root,mode ="label",
                      axis="axial", save_dir=save_dir, resample_labels=False, save_all_slices=False)
    
    img_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/"
    label_root = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/labels/final"
    save_dir = "/data/benchaaben/classifier/ct_overlays/pretrained"
    visualize_dataset(uids, img_root, label_root,mode ="label",
                      axis="axial", save_dir=save_dir, resample_labels=True, save_all_slices=False)
    
    """

     /data/colon_cancer/nnUNet_raw/Dataset102_CC/imagesTr/2.nii.gz -> ❌ missing
     /data/colon_cancer/nnUNet_raw/Dataset102_CC/imagesTr/002.nii.gz -> ❌ missing
     /data/colon_cancer/nnUNet_raw/Dataset102_CC/imagesTr/2_0000.nii.gz -> ❌ missing
     /data/colon_cancer/nnUNet_raw/Dataset102_CC/imagesTr/002_0000.nii.gz -> ✅ exists
     /data/colon_cancer/nnUNet_raw/Dataset102_CC/imagesTr/colon_002.nii.gz -> ❌ missing
     /data/colon_cancer/nnUNet_raw/Dataset102_CC/imagesTr/2.npy -> ❌ missing
     /data/colon_cancer/nnUNet_raw/Dataset102_CC/imagesTr/2.npz -> ❌ missing
     /data/colon_cancer/nnUNet_raw/Dataset102_CC/imagesTr/002.npz -> ❌ missing
     /data/colon_cancer/nnUNet_raw/Dataset102_CC/imagesTr/002.b2nd -> ❌ missing
     /data/colon_cancer/nnUNet_raw/Dataset102_CC/labelsTr/2.nii.gz -> ❌ missing
     /data/colon_cancer/nnUNet_raw/Dataset102_CC/labelsTr/002.nii.gz -> ✅ exists
     /data/colon_cancer/nnUNet_raw/Dataset102_CC/labelsTr/2_0000.nii.gz -> ❌ missing
     /data/colon_can

interactive(children=(IntSlider(value=47, continuous_update=False, description='UID 2', max=93), Output()), _d…

     /data/colon_cancer/nnUNet_preprocessed/Dataset102_CC/nnUNetPlans_3d_fullres/2.nii.gz -> ❌ missing
     /data/colon_cancer/nnUNet_preprocessed/Dataset102_CC/nnUNetPlans_3d_fullres/002.nii.gz -> ❌ missing
     /data/colon_cancer/nnUNet_preprocessed/Dataset102_CC/nnUNetPlans_3d_fullres/2_0000.nii.gz -> ❌ missing
     /data/colon_cancer/nnUNet_preprocessed/Dataset102_CC/nnUNetPlans_3d_fullres/002_0000.nii.gz -> ❌ missing
     /data/colon_cancer/nnUNet_preprocessed/Dataset102_CC/nnUNetPlans_3d_fullres/colon_002.nii.gz -> ❌ missing
     /data/colon_cancer/nnUNet_preprocessed/Dataset102_CC/nnUNetPlans_3d_fullres/2.npy -> ❌ missing
     /data/colon_cancer/nnUNet_preprocessed/Dataset102_CC/nnUNetPlans_3d_fullres/2.npz -> ❌ missing
     /data/colon_cancer/nnUNet_preprocessed/Dataset102_CC/nnUNetPlans_3d_fullres/002.npz -> ❌ missing
     /data/colon_cancer/nnUNet_preprocessed/Dataset102_CC/nnUNetPlans_3d_fullres/002.b2nd -> ✅ exists
     /data/colon_cancer/nnUNet_preprocessed/Dataset102_CC/n

interactive(children=(IntSlider(value=225, continuous_update=False, description='UID 2', max=449), Output()), …

In [ ]:
################# pickle file for metadata from nnunet preprocessing script  ##############################
import pickle
from pathlib import Path 
from tqdm import tqdm
path=f"/data/colon_cancer/nnUNet_preprocessed/Dataset100_CC/nnUNetPlans_3d_fullres/001.pkl"
with open(path, "rb") as f:
    arr = pickle.load(f)
print(arr.keys())
print(arr["shape_after_cropping_and_before_resampling"])
print(arr["shape_before_cropping"])


In [ ]:
################### all labels: total_segmentatot, bowel wall thickening #############################
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import ipywidgets as widgets
from IPython.display import display


from matplotlib.colors import ListedColormap, to_rgba
from matplotlib import cm
from matplotlib.colors import BoundaryNorm
sys.path.append(str(Path.cwd().parent))
from utils.io_utils import load_nifti
from matplotlib.patches import Patch
import os
# --- config ---
#images_dir = Path("/data/colon_cancer/CC_update/image")
images_dir = Path("/data/colon_cancer/nnUNet_raw/Dataset100_CC/imagesTr")

LABEL_MAPS = {
    "tissue_4_types": {
        "background": 0,
        "subcutaneous_fat": 1,
        "torso_fat": 2,
        "skeletal_muscle": 3,
        "intermuscular_fat": 4,
    },
    "colon": {
        "background": 0,
        "colon": 20,
    },
    "Bowel_thickening": {
        "background": 0,
        "bowel_thickening": 1,
    },
}

# --- Input label directories with their corresponding maps ---
LABEL_DIRS_AND_MAPS = [
    #(Path("/data/benchaaben/colon_seg/outputs/tissue_4_types"), LABEL_MAPS["tissue_4_types"]),
    #(Path("/data/colon_cancer/totalseg/total"), LABEL_MAPS["colon"]),
    (Path("/data/benchaaben/colon_seg/outputs/total"), LABEL_MAPS["colon"]),
    #(Path("/data/colon_cancer/totalseg/total"), LABEL_MAPS["colon"]),
    #(Path("/data/colon_cancer/Task101_Colon/raw_splitted/labelsTs"), LABEL_MAPS["Bowel_thickening"]),
    
    #(Path("/data/colon_cancer/CC_update/segs"), LABEL_MAPS["Bowel_thickening"])
]

WINDOW_LEVEL = 50
WINDOW_WIDTH = 350

OVERLAY_ALPHA = 0.30  # slightly higher for clearer overlays

# High-contrast palette (Glasbey-like). Will cycle if you have more labels.
HIGH_CONTRAST_COLORS = [
    "#0000FF", "#FF0000", "#00FF00", "#FF00FF", "#00FFFF", "#FFFF00", "#000000", "#FF8000",
    "#8000FF", "#0080FF", "#80FF00", "#FF0080", "#00FF80", "#808000", "#008000", "#800000",
    "#000080", "#804000", "#408000", "#008040", "#400080", "#804080", "#408080", "#808040",
    "#FF8080", "#80FF80", "#8080FF", "#FF80FF", "#80FFFF", "#FFFF80", "#404040", "#C00000",
]

# --- helpers ---
def window_ct_hu(ct_hu: np.ndarray, level: float = WINDOW_LEVEL, width: float = WINDOW_WIDTH) -> np.ndarray:
    lower = level - width / 2.0
    upper = level + width / 2.0
    ct_clipped = np.clip(ct_hu, lower, upper)
    ct_norm = (ct_clipped - lower) / (upper - lower + 1e-6)
    return ct_norm


def get_image_label_pairs(indices=None, n_random=2):
    rng = np.random.default_rng(0)
    all_cts = sorted(images_dir.glob("*.nii.gz"))
    all_indices = [int(f.name.split("_")[0]) for f in all_cts]
    if indices is None:
        chosen = list(rng.choice(all_indices, size=n_random, replace=False))
    else:
        chosen = indices

    pairs = []
    for idx in chosen:
        ct_file = images_dir / f"{idx:03d}_0000.nii.gz"
        label_files = []
        for label_dir, _ in LABEL_DIRS_AND_MAPS:
            lf = label_dir / f"{idx:03d}.nii.gz"
            label_files.append(lf if lf.exists() else None)
        pairs.append((ct_file, label_files))
    return pairs


def build_distinct_cmap(n_labels: int, alpha: float = OVERLAY_ALPHA) -> ListedColormap:
    # Index 0 is background (transparent). Others draw from HIGH_CONTRAST_COLORS, then HSV fallback.
    colors = np.zeros((n_labels, 4), dtype=float)
    if n_labels == 0:
        return ListedColormap(colors)

    colors[0] = [0, 0, 0, 0]

    num_needed = max(0, n_labels - 1)
    base_rgba = []

    # Use high-contrast palette first (cycled if needed)
    if num_needed > 0:
        if num_needed <= len(HIGH_CONTRAST_COLORS):
            picks = HIGH_CONTRAST_COLORS[:num_needed]
        else:
            # cycle through list, then add HSV distinct hues for overflow
            cycles = [HIGH_CONTRAST_COLORS[i % len(HIGH_CONTRAST_COLORS)] for i in range(num_needed)]
            overflow = num_needed - len(HIGH_CONTRAST_COLORS)
            if overflow > 0:
                hsv_more = cm.hsv(np.linspace(0, 1, overflow, endpoint=False))[:, :3]
                cycles[-overflow:] = [tuple(rgb) for rgb in hsv_more]
            picks = cycles

        for p in picks:
            try:
                base_rgba.append(to_rgba(p, alpha=None))
            except Exception:
                base_rgba.append((0.5, 0.5, 0.5, 1.0))

    if num_needed > 0:
        colors[1:1+num_needed, :3] = np.array(base_rgba)[:, :3]
        colors[1:1+num_needed, 3] = alpha

    return ListedColormap(colors)

def show_interactive_pair(ct_path: Path, label_paths):
    # --- Load CT ---
    ct, _, _, _ = load_nifti(ct_path)
    print(ct.shape)
    # --- Build globally unique remapped labels across all provided label files ---
    # We assign new ids like: 1..K (0 is background). Each source map contributes len(non-bg) ids.
    all_remapped_labels = []
    id_to_name = {}  # global_id -> readable name
    current_id = 1

    for lp, (_, label_map) in zip(label_paths, LABEL_DIRS_AND_MAPS):
        if lp is None or not lp.exists():
            continue

        label_data, _, _, _ = load_nifti(lp)
        label_data = label_data.astype(np.int32)
        

        # Create a remapped array for this file
        remapped = np.zeros_like(label_data, dtype=np.int32)

        # Stable order: iterate label_map keys except background
        non_bg_items = [(name, val) for name, val in label_map.items() if val != 0]

        for name, original_val in non_bg_items:
            remapped[label_data == original_val] = current_id
            id_to_name[current_id] = name
            current_id += 1

        all_remapped_labels.append(remapped)

    n_total_labels = current_id  # includes 0 (background) up to last assigned id
    global_cmap = build_distinct_cmap(n_total_labels)

    # --- Interactive plotting ---
    def plot_slice(slice_idx: int):
        plt.figure(figsize=(6, 6))
        ct_ax = ct[:, :, slice_idx]
        ct_img = window_ct_hu(ct_ax, WINDOW_LEVEL, WINDOW_WIDTH)
        plt.imshow(ct_img.T, cmap="gray", origin="lower")

        # Combine overlays; later label files take precedence where overlapping
        combined_overlay = np.zeros_like(ct_ax, dtype=np.int32)
        for remapped in all_remapped_labels:
            lab_ax = remapped[:, :, slice_idx]
            combined_overlay = np.where(lab_ax > 0, lab_ax, combined_overlay)
        norm = BoundaryNorm(boundaries=np.arange(n_total_labels+1)-0.5, ncolors=n_total_labels)
        # Draw overlay once with the global colormap
        if n_total_labels > 1:
            plt.imshow(combined_overlay.T, cmap=global_cmap, norm=norm, origin="lower", interpolation="nearest")

        # Build legend only for labels present in this slice
        present_ids = set(np.unique(combined_overlay)) - {0}
        legend_elements = []
        for gid in sorted(present_ids):
            name = id_to_name.get(int(gid), f"Label {int(gid)}")
            rgba = global_cmap(norm(int(gid)))  # use same norm as imshow
            legend_elements.append(Patch(facecolor=rgba, edgecolor='k', label=name))

        plt.axis("off")
        plt.title(f"{ct_path.stem} - slice {slice_idx}")
        #os.makedirs(f"/data/benchaaben/classifier/{ct_path.stem}", exist_ok=True)
        #save_path = os.path.join(f"/data/benchaaben/classifier/{ct_path.stem}/pred_{slice_idx}.png")
        #plt.savefig(save_path, bbox_inches='tight', dpi=150)
        plt.show()

        if legend_elements:
            plt.figure(figsize=(max(4, len(legend_elements)), 1.2))
            plt.legend(handles=legend_elements, loc='center', ncol=len(legend_elements), frameon=False)
            plt.axis('off')
            plt.tight_layout()
            plt.show()

    slice_slider = widgets.IntSlider(
        value=ct.shape[2] // 2,
        min=0,
        max=ct.shape[2] - 1,
        step=1,
        description=f"Slice {ct_path.stem}:",
        continuous_update=False,
    )
    display(widgets.interact(plot_slice, slice_idx=slice_slider))


def show_images(indices=None, n_random=2):
    pairs = get_image_label_pairs(indices=indices, n_random=n_random)
    for ct_path, label_paths in pairs:
        show_interactive_pair(ct_path, label_paths)


# --- Examples ---
# show_images(n_random=2)
show_images(indices=[2])
#show_images(indices=[18,29,300])